## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_openrouter import ChatOpenRouter

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gpt-4.1-nano"
model = "nvidia/nemotron-3.5-lightning:free"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOpenRouter(temperature=0, model_name=model)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='844d8170-a3f6-4666-b59c-d3af1e485ea8', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [9]:
llm = ChatOpenRouter(temperature=0, model_name=model, timeout=120)
response = llm.invoke([HumanMessage(content="Who is Avery?")])
response.content
#llm.invoke("Who is Avery?")

'"Avery" can refer to several different people, places, or things depending on context. Here are the most common meanings:\n\n**As a name:**\n- **Given name:** An English-origin name used for both boys and girls, derived from an Old French surname meaning "elf counsel" or "ruler of elves." It\'s been increasingly popular in the U.S. for several decades.\n- **Surname:** A French-derived last name, ultimately from the personal name *Alberic* (meaning "elf ruler").\n\n**Notable people:**\n- **Avery Johnson** – Former NBA player and coach, notably of the Dallas Mavericks\n- **Avery Brooks** – Actor best known as Benjamin Sisko in *Star Trek: Deep Space Nine*\n- **Avery Bradley** – Professional basketball player\n- **Avery Singer** – Contemporary visual artist\n- **Avery Schreiber** – Comedian and actor (1930–2002)\n- **Avery Johnson** (different from the coach) – Various athletes and public figures\n\n**Fictional characters:**\n- **Avery Johnson** – The Spartan-II commander in the *Halo* v

## Time to put this together!

In [10]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [12]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [14]:
answer_question("Who is Averi Lancaster?", [])

'Based on the information provided about Insurellm, **Avery Lancaster** (the context lists the name as "Avery," and you mentioned "Averi") is the Co-Founder and Chief Executive Officer of the company. Here are the key details:\n\n- **Date of Birth:** March 15, 1985  \n- **Location:** San Francisco, California  \n- **Current Salary:** $225,000  \n- **Role:** Co-Founder & CEO since 2015, having guided Insurellm to become a leading Insurance Tech provider in the mainstream market.  \n- **Background:** Previously a Senior Product Manager at Innovate Insurance Solutions (2013–2015), where she developed insurance products tailored to the tech sector.  \n\nAvery is recognized for her innovative leadership strategies and risk management expertise that have been instrumental in Insurellm\'s growth and market positioning. \n\nIf you\'d like more details about her career progression, leadership style, or specific initiatives she\'s overseen, just let me know!'

In [16]:
llm2 = ChatOllama(temperature=0, model="phi4:latest")

In [17]:
def answer_question_ollama(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm2.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [18]:
answer_question("Who is Averi Lancaster?", [])

'(If you\'re referring to **Avery Lancaster** — the spelling in our records is with an "e" — here\'s what we have on file:\n\n**Avery Lancaster** is a co-founder and executive at Insurellm.  \n- **Role:** Co-Founder & Chief Executive Officer (CEO)  \n- **Date of Birth:** March 15, 1985  \n- **Location:** San Francisco, California  \n- **Current Salary:** $225,000  \n- **Career History:** \n  - Co-Founder & CEO of Insurellm (2015 – Present), leading the company to become a recognized Insurance Tech provider.  \n  - Senior Product Manager at Innovate Insurance Solutions (2013 – 2015), where she helped design insurance products for the tech sector.  \n\nAvery is known for her innovative leadership strategies, risk management expertise, and for guiding Insurellm into the mainstream insurance market.\n\n*Note: The context also includes a separate entry about a "Senior Data Engineer" and someone named Maxine, which appears to be a misplaced or mixed record. If you were asking about a differe

## What could possibly come next? 😂

In [19]:
gr.ChatInterface(answer_question).launch()

c:\Code\LLM Engineering\Code\code_my_github\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!